# OPTIONAL - NEEDED FOR SETUP

In [ ]:
# ! pip install tensorflow matplotlib numpy scikit-learn

In [ ]:
# FOR CHECKING TENSORFLOW INSTALLATION --> YOU SHOULD SEE A TENSOR WITHOUT ANY ERROR
# import tensorflow as tf
# output = tf.reduce_sum(tf.random.normal([100, 100]))
# if(str(type(output)) == "<class 'tensorflow.python.framework.ops.EagerTensor'>"):
#     print("TensorFlow is installed correctly and working!")
# else:    
#     print("There seems to be an issue with your TensorFlow installation.")


# Autoencoder — Working Principles

- **Dataset:** `fashion_mnist` images (28×28), normalized to [0,1].
- **Model:** Keras `Model` subclass separating `encoder` and `decoder`.
- **Encoder:** `Flatten()` → `Dense(latent_dim, activation='relu')` compresses to a `latent_dim` bottleneck (64).
- **Bottleneck:** compact latent vector `z` that captures salient features.
- **Decoder:** `Dense(784, activation='sigmoid')` → `Reshape((28,28))` reconstructs images.
- **Training:** inputs used as targets; `optimizer='adam'`, `loss=MeanSquaredError()` to minimize reconstruction error.
- **Notes:** dense autoencoder (no convs); latent size trades off compression vs. fidelity; random sampling is meaningful only with structured/regularized latent spaces.

This cell summarizes the architecture and training behavior demonstrated in the notebook.

# 1. Setup and Data Preparation
First, we import the necessary libraries and load the dataset. We normalize the pixel values to a range between 0 and 1 for better training stability.



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, losses
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Model

# Load the dataset (we don't need the labels for training an autoencoder)
(x_train, _), (x_test, _) = fashion_mnist.load_data()

# Normalize and reshape
x_train = x_train.astype('float32') / ???        # Normalize to [0, 1]
x_test = x_test.astype('float32') / ???          # Normalize to [0, 1]

print(f"Training shape: {x_train.shape}") # (60000, 28, 28)

# 2. Define the Autoencoder Architecture
We'll use a Subclassing API approach. This allows us to separate the encoder (compression) and decoder (reconstruction) into two distinct components.

In [ ]:
class Autoencoder(Model):
  def __init__(self, latent_dim):
    super(Autoencoder, self).__init__()
    self.latent_dim = latent_dim

    # Encoder: Flatten 28x28 image to 784, then compress to latent_dim
    self.encoder = tf.keras.Sequential([
      layers.Flatten(),
      layers.Dense(???, activation='relu'),      # Compress to latent_dim
    ])

    # Decoder: Expand back to 784, then reshape to 28x28
    self.decoder = tf.keras.Sequential([
      layers.Dense(???, activation='sigmoid'),    # Expand back to 784 (28*28)
      layers.Reshape((???, ???))                  # Reshape to 28x28
    ])

  def call(self, x):
    encoded = self.encoder(x)           # Pass input through encoder
    decoded = self.decoder(encoded)     # Pass encoded representation through decoder            
    return decoded

# Initialize the model with a bottleneck of 64 dimensions and set loss as mean squared error and optimizer as Adam
latent_dim = 64
autoencoder = Autoencoder(latent_dim)
autoencoder.compile(optimizer=???, loss=???)

# IF YOU WANT TO SEE THE MODEL SUMMARY, UNCOMMENT THE LINES BELOW
# autoencoder(x_train)  # Build the model by calling it once
# autoencoder.encoder.summary()
# autoencoder.decoder.summary()

# 3. Training the Model
In an autoencoder, the input is the target. We want the model to learn to output exactly what we gave it.


In [ ]:
autoencoder.fit(???, ???,    # Fill in with input and target 
                epochs=2,   # YOU CAN INCREASE THIS FOR BETTER RECONSTRUCTION
                shuffle=True,
                validation_data=(x_test, x_test))

# 4. Visualizing the Results
To evaluate the model, we compare the original images from the test set with their reconstructed versions produced by the autoencoder.


In [ ]:
# Pass the test images through the autoencoder
encoded_imgs = autoencoder.encoder(???).numpy() # Pass test images through encoder
reconstructed_imgs = autoencoder.decoder(encoded_imgs).numpy()

n = 10
plt.figure(figsize=(20, 6)) # Increased height for the third row
for i in range(n):
  # Display Original
  ax = plt.subplot(3, n, i + 1) # Changed to 3 rows
  plt.imshow(???)               # Display i-th original image
  plt.title("Original")
  plt.gray()
  ax.get_xaxis().set_visible(False)
  ax.get_yaxis().set_visible(False)

  # Display Latent Vector Heatmap
  ax = plt.subplot(3, n, i + 1 + n) # Second row for latent vector
  # Reshape latent vector for better visualization if possible (e.g., 64 -> 8x8)
  if encoded_imgs.shape[1] == 64:
      latent_display = encoded_imgs[i].reshape(???, ???) # Reshape to 8x8 for better visualization
  else:
      latent_display = encoded_imgs[i].reshape(1, -1) # Display as a single row if not square
  plt.imshow(??? , cmap='viridis') # What to show here ?
  plt.title("Latent")
  ax.get_xaxis().set_visible(False)
  ax.get_yaxis().set_visible(False)

  # Display Reconstruction
  ax = plt.subplot(3, n, i + 1 + 2*n) # Third row for reconstruction
  plt.imshow(???)              # Display i-th reconstructed image
  plt.title("Reconstructed")
  plt.gray()
  ax.get_xaxis().set_visible(False)
  ax.get_yaxis().set_visible(False)
plt.show()

# See the t-SNE visualization of 100 random test samples

In [ ]:
from sklearn.manifold import TSNE

# Load test labels for coloring (images are already in x_test)
(_, _), (_, y_test) = fashion_mnist.load_data()

# Select 1000 random test samples
num_samples = 1000
sample_idx = np.random.choice(x_test.shape[0], size=num_samples, replace=False)
x_sample = x_test[sample_idx]
y_sample = y_test[sample_idx]

# Encode samples to latent space
z_sample = autoencoder.encoder(x_sample).numpy()

# t-SNE on latent vectors
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init="pca", learning_rate="auto")
z_tsne = tsne.fit_transform(z_sample)

# Plot
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

plt.figure(figsize=(8, 6))
sc = plt.scatter(z_tsne[:, 0], z_tsne[:, 1], c=y_sample, cmap="tab10", s=45, alpha=0.85)
cbar = plt.colorbar(sc, ticks=range(10))
cbar.ax.set_yticklabels(class_names)
plt.title("t-SNE of 100 Random Test Samples (Latent Space)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.show()

# Can we generate realistic outputs from random inputs ?
Below, we generate 10 random latent vectors, visualize their heatmaps, and display the images produced by passing these latent vectors through the decoder.

In [ ]:
n = 10
plt.figure(figsize=(20, 6))

# Generate 10 random latent vectors
random_latent_vectors = tf.random.uniform(shape=(n, latent_dim))

decoded_imgs = autoencoder.decoder(random_latent_vectors).numpy()

for i in range(n):
  # Display Latent Vector Heatmap
  ax = plt.subplot(2, n, i + 1)
  # Reshape latent vector for better visualization (e.g., 64 -> 8x8)
  if latent_dim == 64:
      latent_display = random_latent_vectors[i].numpy().reshape(8, 8)
  else:
      latent_display = random_latent_vectors[i].numpy().reshape(1, -1) # Display as a single row if not square
  plt.imshow(???, cmap='viridis') # Display the random latent vector as a heatmap
  plt.title(f"Latent {i+1}")
  ax.get_xaxis().set_visible(False)
  ax.get_yaxis().set_visible(False)

  # Display Generated Image
  ax = plt.subplot(2, n, i + 1 + n)
  plt.imshow(???) # Display the generated image from the random latent vector
  plt.title(f"Generated {i+1}")
  plt.gray()
  ax.get_xaxis().set_visible(False)
  ax.get_yaxis().set_visible(False)

plt.show()